In [1]:
import os
os.environ["OLLAMA_HOST"]="10.103.12.94:11434"

In [2]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
import numpy as np
import langchain  # for legacy llm_cache

# LangChain community integrations
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyPDFLoader,
)
from langchain_community.cache import SQLiteCache

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# ---------------------------
# Config
# ---------------------------
PERSIST_DIR     = "./faiss_db_llama32_nomic"
DATA_DIR        = "./data"

# Ollama models
EMBED_MODEL     = "nomic-embed-text"
CHAT_MODEL      = "gemma3:1b"
OLLAMA_BASE_URL = "http://10.103.12.94:11434"

# LLM cache (legacy API your version supports)
LLM_CACHE_PATH = "./llm_cache.sqlite"
Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)
Path(LLM_CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)
langchain.llm_cache = SQLiteCache(database_path=LLM_CACHE_PATH)


# ---------------------------
# Helpers
# ---------------------------
def load_docs():
    """Load .txt, .md and .pdf files from DATA_DIR."""
    docs = []
    data_path = Path(DATA_DIR)

    if data_path.exists():
        # Text files
        txt_loader = DirectoryLoader(
            DATA_DIR,
            glob="**/*.txt",
            loader_cls=TextLoader,
            show_progress=True,
            use_multithreading=True,
        )
        md_loader = DirectoryLoader(
            DATA_DIR,
            glob="**/*.md",
            loader_cls=TextLoader,
            show_progress=True,
            use_multithreading=True,
        )

        try:
            docs.extend(txt_loader.load())
        except Exception as e:
            print(f"Error loading .txt files: {e}")

        try:
            docs.extend(md_loader.load())
        except Exception as e:
            print(f"Error loading .md files: {e}")

        # PDFs
        for pdf in data_path.rglob("*.pdf"):
            try:
                print(f"Loading PDF: {pdf}")
                docs.extend(PyPDFLoader(str(pdf)).load())
            except Exception as e1:
                print(f"Error loading PDF {pdf}: {e1}")

    # Fallback docs if nothing loaded
    if not docs:
        from langchain.schema import Document
        docs = [
            Document("LangChain example doc", {"source": "fallback"}),
            Document("FAISS sample doc", {"source": "fallback"}),
        ]
    return docs


def build_or_load_faiss(embeddings):
    """Create/load a persistent FAISS index."""
    index_file = Path(PERSIST_DIR) / "faiss.index"
    store_file = Path(PERSIST_DIR) / "faiss.pkl"

    if index_file.exists() and store_file.exists():
        print("Loading existing FAISS index...")
        return FAISS.load_local(
            PERSIST_DIR,
            embeddings=embeddings,
            allow_dangerous_deserialization=True,
        )

    print("Building new FAISS index...")
    docs = load_docs()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    vs = FAISS.from_documents(chunks, embeddings)
    vs.save_local(PERSIST_DIR)
    return vs


def make_rag_chain(retriever, llm):
    """RAG chain: retriever → prompt → LLM → text."""
    def format_docs(docs):
        return "\n\n".join(
            f"Source: {d.metadata.get('source','?')}\n{d.page_content}"
            for d in docs
        )

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a concise, helpful assistant. Use the provided context to answer. "
         "If the answer isn't in the context, say so.\n\nContext:\n{context}"),
        ("human", "{question}")
    ])

    chain = (
        {
            "context": retriever | (lambda docs: format_docs(docs)),
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


def timed(fn):
    """Time a callable."""
    def _inner(*args, **kwargs):
        t0 = time.time()
        out = fn(*args, **kwargs)
        return out, time.time() - t0
    return _inner


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1D vectors."""
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


# ---------------------------
# Main
# ---------------------------
if __name__ == "__main__":
    # 1) Embeddings (Ollama)
    embeddings = OllamaEmbeddings(
        model=EMBED_MODEL,
        base_url=OLLAMA_BASE_URL,
    )

    # 2) FAISS index
    vstore = build_or_load_faiss(embeddings)
    retriever = vstore.as_retriever(search_kwargs={"k": 4})

    # 3) LLM (Ollama)
    llm = ChatOllama(
        model=CHAT_MODEL,
        temperature=0,
        base_url=OLLAMA_BASE_URL,
    )

    # 4) RAG chain
    chain = make_rag_chain(retriever, llm)

    # 5) Semantic cache data structures
    semantic_cache_questions = []   # list[str]
    semantic_cache_answers = []     # list[str]
    semantic_cache_vectors = []     # list[np.ndarray]

    SEMANTIC_THRESHOLD = 0.9  # adjust if needed (0.8 more aggressive reuse)

    def semantic_cached_query(question: str) -> str:
        """Return cached answer if question is semantically similar to previous ones."""
        # Embed incoming question
        q_vec = np.array(embeddings.embed_query(question), dtype=float)

        # If cache not empty, search for nearest neighbour
        if semantic_cache_vectors:
            sims = [
                cosine_similarity(q_vec, v) for v in semantic_cache_vectors
            ]
            max_idx = int(np.argmax(sims))
            max_sim = sims[max_idx]

            # If similar enough, reuse that answer
            if max_sim >= SEMANTIC_THRESHOLD:
                # print(f"[Semantic cache hit] similarity={max_sim:.3f}")
                return semantic_cache_answers[max_idx]

        # If no hit: run full RAG chain
        ans = chain.invoke(question)

        # Store in semantic cache
        semantic_cache_questions.append(question)
        semantic_cache_answers.append(ans)
        semantic_cache_vectors.append(q_vec)

        return ans

    # 6) Run evaluation over your CSV
    csv_path = r"C:\Users\surya.adatravu\Documents\RAGAnalysis\RA_FSM_QA.csv"
    df = pd.read_csv(csv_path)

    df["t0"] = 0.0
    df["t1"] = 0.0
    df["ans0"] = ""
    df["ans1"] = ""

    for i in range(df.shape[0]):
        q = df.loc[i, "Question"]

        # First run (likely miss initially)
        ans1, t1 = timed(semantic_cached_query)(q)
        df.loc[i, "t0"] = t1
        df.loc[i, "ans0"] = ans1

        # Second run (same question → semantic hit for sure)
        ans2, t2 = timed(semantic_cached_query)(q)
        df.loc[i, "t1"] = t2
        df.loc[i, "ans1"] = ans2

        print(f"Row {i} done | t0={t1:.3f}s t1={t2:.3f}s")

    df.to_csv("results_llama32_nomic_semantic_cache.csv", index=False)

    print("\nArtifacts:")
    print(f"FAISS DB:   {Path(PERSIST_DIR).resolve()}")
    print(f"LLM Cache:  {Path(LLM_CACHE_PATH).resolve()}")


Building new FAISS index...


0it [00:00, ?it/s]
0it [00:00, ?it/s]

Loading PDF: data\RA_FSM_Paper 2.pdf


Row 0 done | t0=7.009s t1=0.554s
Row 1 done | t0=6.709s t1=0.484s
Row 2 done | t0=5.914s t1=0.443s
Row 3 done | t0=8.220s t1=0.467s
Row 4 done | t0=5.592s t1=0.552s
Row 5 done | t0=5.969s t1=0.465s
Row 6 done | t0=5.445s t1=0.472s
Row 7 done | t0=6.682s t1=0.450s
Row 8 done | t0=5.565s t1=0.494s
Row 9 done | t0=5.725s t1=0.453s
Row 10 done | t0=6.980s t1=0.441s
Row 11 done | t0=5.208s t1=0.444s
Row 12 done | t0=5.895s t1=0.447s
Row 13 done | t0=6.294s t1=0.423s
Row 14 done | t0=5.456s t1=0.470s
Row 15 done | t0=6.649s t1=0.475s
Row 16 done | t0=5.271s t1=0.450s
Row 17 done | t0=6.843s t1=0.459s
Row 18 done | t0=6.556s t1=0.455s
Row 19 done | t0=6.357s t1=0.446s
Row 20 done | t0=9.552s t1=0.459s
Row 21 done | t0=5.844s t1=0.469s
Row 22 done | t0=7.590s t1=0.437s
Row 23 done | t0=6.213s t1=0.491s
Row 24 done | t0=5.444s t1=0.447s
Row 25 done | t0=5.431s t1=0.453s
Row 26 done | t0=5.978s t1=0.464s
Row 27 done | t0=5.892s t1=0.477s
Row 28 done | t0=9.872s t1=0.446s
Row 29 done | t0=10.778s